# FlashAttentionCUDA — Nível 1: kernel com tiling + softmax online

Checklist deste notebook (Nível 1):

1. Confirmar GPU disponível.
2. Clonar o repositório e compilar a extensão `flash_attention_tiled`.
3. Rodar o harness de corretude (kernel tiled vs. referência *e* vs. kernel ingênuo).
4. Rodar o benchmark mínimo (tempo + memória de pico, ingênuo vs. tiled vs. PyTorch nativo, variando N).

Pré-requisito: o notebook `00_setup_and_naive.ipynb` já deve ter sido executado com sucesso.

## 1. GPU disponível

In [ ]:
!nvidia-smi

## 2. Clonar o repositório e compilar

In [ ]:
!git clone https://github.com/IagoNat/HPC.git
%cd HPC/FlashAttentionCUDA
!pip install -q ninja
!rm -rf /root/.cache/torch_extensions  # evita cache de build travado de sessoes anteriores

In [ ]:
# compila a extensão tiled (primeira chamada faz o build e já roda um teste mínimo)
import torch
from python.flash_attention_tiled import flash_attention_tiled

Q = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
K = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
V = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
out = flash_attention_tiled(Q, K, V)
print("compilou e rodou - shape de saída:", out.shape)

## 3. Harness de corretude

Compara o kernel tiled contra (a) a referência PyTorch e (b) o kernel ingênuo do Nível 0. Inclui N menor, igual e maior que o tile size (`BLOCK_N = 32`), pra exercitar 1 tile só e múltiplos tiles (inclusive tile final parcial).

In [ ]:
!python tests/test_tiled_correctness.py

## 4. Benchmark mínimo

Tempo (eventos CUDA) e memória de pico, variando N, comparando as três versões: ingênuo, tiled e PyTorch nativo.

In [ ]:
!python python/benchmark.py

## Status do Nível 1

- [ ] Extensão `flash_attention_tiled` compila e roda na T4.
- [ ] Harness de corretude passa em todos os casos (vs. referência e vs. kernel ingênuo).
- [ ] Benchmark mínimo roda sem erro.
- [ ] Kernel tiled usa menos memória de pico que o ingênuo nos N maiores (esperado, é o ponto central do projeto).